In [5]:
# ===== Environment Setup =====

!pip install -q \
chromadb==1.5.9 \
sentence-transformers \
mitreattack-python \
transformers \
accelerate

import sys

sys.path.insert(
    0,
    "/kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data"
)

!cp /kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data/enterprise-attack.json .
!cp /kaggle/input/datasets/karthiksethuraman6/qwen-atlas-data/index_mappings.json .

print("Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 78.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.4/574.4 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 117.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 90.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [6]:
# ===== Retriever Setup =====

import chroma_rag

print("Retriever loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retriever loaded


In [7]:
# ===== Build ChromaDB =====

chroma_rag.ingest_techniques()
chroma_rag.ingest_groups()
chroma_rag.ingest_mitigations()

print("Collection Count:", chroma_rag.col.count())

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Ingested 697 techniques


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Ingested 174 groups


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Ingested 44 mitigations
Collection Count: 915


In [8]:
# ===== Load Model =====

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

import torch

MODEL = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model Loaded")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model Loaded


In [9]:
# ===== Plain Inference =====

def plain_inference(query, max_tokens=250):

    messages = [
        {"role": "user", "content": query}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens
    )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

In [10]:
# ===== RAG Inference =====

def rag_inference(query, max_tokens=250):

    results = chroma_rag.smart_retrieve(query)

    context = "\n\n".join(
        r["text"][:1500]
        for r in results
    )

    prompt = f"""
You are a cybersecurity threat intelligence assistant.

Answer using ONLY the supplied ATT&CK context.

Do not use external knowledge.

If the answer is not present in the context,
explicitly state that it is unavailable.

ATT&CK Context:
{context}

Question:
{query}

Answer:
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens
    )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

In [11]:
TEST_QUERIES = [
    "What is T1003?",
    "Explain T1055.",
    "What techniques does APT29 use?",
    "What credential access techniques does APT29 use?",
    "What is APT33?",
    "How can T1003 be mitigated?",
    "Which groups use T1003?",
    "What is Process Injection?",
    "What platforms are affected by T1003?",
    "What is T1195.002?"
]

In [12]:
plain_results = {}
rag_results = {}

for query in TEST_QUERIES:

    print("=" * 80)
    print(query)
    print("=" * 80)

    plain_results[query] = plain_inference(query)
    rag_results[query] = rag_inference(query)

    print("\nPLAIN:\n")
    print(plain_results[query])

    print("\nRAG:\n")
    print(rag_results[query])

    print("\n\n")

What is T1003?
[Router] Technique ID lookup

PLAIN:

T1003 is a tactic, technique, and procedure (TTP) identifier in the MITRE ATT&CK framework. ATT&CK stands for "Adversarial Tactics, Techniques, and Common Knowledge," which is a knowledge base of adversary tactics and techniques based on real-world observations.

In the context of cybersecurity, T1003 specifically refers to:

**T1003: Network Service Scanning**

This tactic involves adversaries scanning network services to gather information about open ports, services, and protocols that are available on the target network. This can help them identify potential vulnerabilities or misconfigurations that could be exploited.

The purpose of this technique is to understand the network environment, discover open services, and potentially identify weaknesses that can be leveraged for further attacks.

If you need more detailed information or specific examples related to T1003, feel free to ask!

RAG:

T1003 is a technique in the MITRE ATT&